## Imports and Configuration

In [ ]:
# Library imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.cluster import KMeans

In [ ]:
# Plot aesthetics configuration
sns.set_theme(style="whitegrid")

In [ ]:
# Random seed configuration
SEED = 1234
np.random.seed(SEED)
print(f"Random seed: {SEED}")

## Loading and Initial Data Overview

In [ ]:
# Loading the dataset including the separator
raw_data = pd.read_csv('student-porB.csv', sep=';')

In [ ]:
# Visual preview of the first records of the dataset
raw_data.head()

In [ ]:
# Basic inspection of data structure and consistency
print(f"Dataset dimensions: {raw_data.shape[0]} rows, {raw_data.shape[1]} columns")
print("\nData types and missing values:")
print(raw_data.info())

In [ ]:
# Statistical analysis of numerical and categorical variables
display(raw_data.describe(include='all'))

In [ ]:
# Verification of uniqueness of values in text columns (checking for typos/errors)
print("\nVerification of unique values for categorical variables:")
for col in raw_data.select_dtypes(include='object'):
    print(f"{col}: {raw_data[col].unique()}")

In [ ]:
# Checking the correctness of ranges for numerical variables
print("\nRanges of numerical variables:")
for col in raw_data.select_dtypes(include=[np.number]):
    print(f"{col:12}: min={raw_data[col].min():2}, max={raw_data[col].max():2}")

## Data Cleaning

In [ ]:
# Creating a working copy and removing predictors G1 and G2 according to project guidelines
df_processed = raw_data.copy().drop(['G1', 'G2'], axis=1)

In [ ]:
# Mapping binary variables to numerical format (0/1)
# Manual mapping ensures full control over the interpretation of variable direction
binary_features = {
    'schoolsup': {'yes': 1, 'no': 0}, 'famsup': {'yes': 1, 'no': 0},
    'paid': {'yes': 1, 'no': 0}, 'activities': {'yes': 1, 'no': 0},
    'nursery': {'yes': 1, 'no': 0}, 'higher': {'yes': 1, 'no': 0},
    'internet': {'yes': 1, 'no': 0}, 'romantic': {'yes': 1, 'no': 0},
    'school': {'GP': 1, 'MS': 0}, 'sex': {'F': 1, 'M': 0},
    'address': {'U': 1, 'R': 0}, 'famsize': {'GT3': 1, 'LE3': 0},
    'Pstatus': {'T': 1, 'A': 0}
}

for col, mapping in binary_features.items():
    df_processed[col] = df_processed[col].map(mapping)

In [ ]:
# Splitting into training and test sets before categorical transformations
# This avoids information leakage about rare categories from the test set
train_raw, test_raw = train_test_split(df_processed, test_size=0.2, random_state=SEED)

In [ ]:
# Transforming remaining nominal variables using One-Hot Encoding
train = pd.get_dummies(train_raw)
test = pd.get_dummies(test_raw)

In [ ]:
# Synchronizing columns between sets (filling missing categories with zeros)
test = test.reindex(columns=train.columns, fill_value=0)

In [ ]:
print(f"Data preparation completed.")
print(f"Training set size: {train.shape}")
print(f"Test set size:    {test.shape}")

## Exploratory Data Analysis (only on the training set)

In [ ]:
# Analysis performed exclusively on the training set to avoid being influenced by test data

In [ ]:
# Visualization of target variable distribution
plt.figure(figsize=(8, 4))
sns.countplot(x='G3', data=train, palette='viridis', hue='G3', legend=False)
plt.title('Distribution of final grades (G3) in the training set')
plt.show()

In [ ]:
# General analysis of interdependencies between variables after factorization
plt.figure(figsize=(12, 10))
correlation_matrix_full = train.apply(lambda x: pd.factorize(x)[0]).corr()
sns.heatmap(correlation_matrix_full, annot=False, cmap='coolwarm')
plt.title('Correlation map of all variables (encoded data)')
plt.show()

In [ ]:
# Analysis of linear correlation of numerical variables with the target variable G3
plt.figure(figsize=(15, 10))
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_matrix = train[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0, annot_kws={"size": 8})
plt.title('Correlation matrix of numerical features relative to G3')
plt.show()

In [ ]:
# Examining the impact of key categorical and discrete variables on the final grade
important_features = ['failures', 'Medu', 'higher', 'studytime']
plt.figure(figsize=(15, 10))
for i, col in enumerate(important_features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x=col, y='G3', data=train, palette='Set2', hue=col, legend=False)
    plt.title(f'Distribution of G3 depending on: {col}')
plt.tight_layout()
plt.show()

## Preparation for Modeling

In [ ]:
# Separation of target variable from predictors for both datasets
X_train = train.drop('G3', axis=1)
X_test = test.drop('G3', axis=1)
y_train = train['G3']
y_test = test['G3']

In [ ]:
def evaluate_model(y_true, y_pred_labels, y_pred_raw, model_name):
    """
    Universal function for model evaluation according to project requirements.
    Calculates Accuracy, accuracy with a margin of +/- 1, and MAE error.
    """
    acc = accuracy_score(y_true, y_pred_labels)
    acc_plus_minus_1 = np.mean(np.abs(y_true - y_pred_labels) <= 1)
    mae = mean_absolute_error(y_true, y_pred_raw)
    
    print(f"--- Results for model: {model_name} ---")
    print(f"Accuracy:                   {acc:.4f}")
    print(f"Accuracy (Margin +/- 1):    {acc_plus_minus_1:.4f}")
    print(f"Mean Absolute Error MAE:    {mae:.4f}\n")
    
    return [acc, acc_plus_minus_1, mae]

## Feature Engineering

In [ ]:
# Standardization of input data - essential for linear regression and k-means algorithms
scaler = StandardScaler()

In [ ]:
# Fitting the scaler to the training data and transforming both datasets
# Maintaining test set isolation (fit only on train)
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns
)

print(f"Scaling completed successfully. Number of input features: {X_train_scaled.shape[1]}")

## Modeling - Linear Regression (Task: Estimation)

In [ ]:
# Initialization and fitting of the regression model to the scaled data
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

In [ ]:
# Generating continuous predictions and rounding them to integers (according to instructions)
y_pred_reg_raw = lin_reg.predict(X_test_scaled)
y_pred_reg_rounded = np.round(y_pred_reg_raw).astype(int)

In [ ]:
# Evaluation of the model using the prepared function
print("LINEAR REGRESSION MODEL EVALUATION (Estimation)")
results_lin = evaluate_model(y_test, y_pred_reg_rounded, y_pred_reg_raw, "Linear Regression")

In [ ]:
# --- Verification of regression model assumptions ---
plt.figure(figsize=(12, 5))

In [ ]:
# Analysis of normality of residuals distribution (Q-Q Plot)
plt.subplot(1, 2, 1)
residuals = y_test - y_pred_reg_raw
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot: Normality of residuals")

In [ ]:
# Examining homoscedasticity (constancy of error variance)
plt.subplot(1, 2, 2)
plt.scatter(y_pred_reg_raw, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted values')
plt.ylabel('Residuals')
plt.title('Residual plot: Homoscedasticity')
plt.show()

In [ ]:
# --- NUMERICAL MODEL DIAGNOSTICS ---

# 1. Test of normality of residuals (Shapiro-Wilk)
# H0: Residuals have a normal distribution. If p < 0.05, we reject normality.
shapiro_p = stats.shapiro(residuals)[1]
print(f"Shapiro-Wilk test (p-value): {shapiro_p:.4f}")
if shapiro_p < 0.05:
    print("Conclusion: Residuals DO NOT have a normal distribution (common with grades 0-20).")
else:
    print("Conclusion: No grounds to reject the normality of residuals.")

# 2. Model coefficients (Interpretation)
weights = pd.DataFrame({
    'Feature': X_train.columns,
    'Weight (Coeff)': lin_reg.coef_
}).sort_values(by='Weight (Coeff)', ascending=False)

print("\nTop 5 features increasing the predicted G3 grade:")
display(weights.head(5))

print("\nTop 5 features decreasing the predicted G3 grade:")
display(weights.tail(5))

# 3. Checking Multicollinearity (simpler method - correlation)
# Instead of VIF, we check if input features are too strongly correlated (above 0.8)
corr_matrix_X = X_train.corr().abs()
upper = corr_matrix_X.where(np.triu(np.ones(corr_matrix_X.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

if len(to_drop) > 0:
    print(f"\nWarning: Highly correlated features detected: {to_drop}")
else:
    print("\nNo strong multicollinearity between features (all r < 0.8).")

In [ ]:
from statsmodels.stats.stattools import durbin_watson

# --- ANALYSIS OF ERROR DISTRIBUTION AND AUTOCORRELATION ---

plt.figure(figsize=(10, 5))

# Histogram of residuals with overlaid density curve (Visual Normality)
sns.histplot(residuals, kde=True, color='royalblue', bins=20)
plt.title('Distribution of residuals (Histogram with KDE curve)')
plt.xlabel('Error value (Residual)')
plt.ylabel('Frequency')
plt.show()

# Durbin-Watson test (Autocorrelation of residuals)
# A result close to 2.0 indicates no autocorrelation. The range [1.5 - 2.5] is acceptable.
dw_stat = durbin_watson(residuals)
print(f"Durbin-Watson statistic: {dw_stat:.4f}")
if 1.5 <= dw_stat <= 2.5:
    print("Conclusion: No significant autocorrelation of residuals (assumption of independence of errors met).")
else:
    print("Conclusion: Potential autocorrelation of residuals detected.")

In [ ]:
# --- IDENTIFICATION OF INFLUENTIAL OBSERVATIONS (COOK'S DISTANCE) ---

from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

# Data preparation (adding constant to the feature matrix)
X_constant = add_constant(X_train_scaled)

# Fitting the OLS (Ordinary Least Squares) model to the training data
model_sm = OLS(y_train.values, X_constant).fit()

# Calculating Cook's distance (measure of influence of individual observations)
influence = model_sm.get_influence()
cooks_d = influence.cooks_distance[0]

# Visualization of Cook's distance
plt.figure(figsize=(10, 4))
plt.stem(np.arange(len(cooks_d)), cooks_d, markerfmt=",")
plt.title("Cook's Distance (Influence of observations on model parameters)")
plt.xlabel('Observation index in the training set')
plt.ylabel('Distance Value')
plt.axhline(y=4/len(X_train), color='red', linestyle='--', label='Critical threshold (4/n)')
plt.legend()
plt.show()

# Detecting indices of observations exceeding the threshold
outlier_indices = np.where(cooks_d > 4/len(X_train))[0]
print(f"Number of high-influence observations (outliers): {len(outlier_indices)}")

In [ ]:
# --- PRECISE Q-Q PLOT (STABLE VERSION) ---

from statsmodels.graphics.gofplots import ProbPlot

# Generating chart data
pp = ProbPlot(residuals, fit=True)

# Creating the figure
fig = pp.qqplot(line='45', alpha=0.5, color='royalblue', lw=1)

plt.title('Quantile-Quantile Plot (Q-Q Plot) - Residuals')
plt.grid(True, alpha=0.3)
plt.show()

# Interpretation: 
# The "drop" of points in the lower-left corner (distribution tails) confirms what the Shapiro-Wilk test showed:
# the model has difficulty correctly estimating very low scores (so-called fat tail).

## Modeling - XGBoost (Task: Classification)

In [ ]:
# Encode labels to a format accepted by XGBoost (range 0 to n_classes-1)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

In [ ]:
# Configuration and training of the gradient classifier
xgb_clf = XGBClassifier(
    random_state=SEED, 
    use_label_encoder=False, 
    eval_metric='logloss'
)
xgb_clf.fit(X_train, y_train_encoded)

In [ ]:
# Class prediction and return to original grade scale (G3)
y_pred_xgb_encoded = xgb_clf.predict(X_test)
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_encoded)

In [ ]:
# Evaluation of the classification model
print("XGBOOST MODEL EVALUATION (Classification)")
# In the case of classification, y_pred_raw is identical to y_pred_labels
results_xgb = evaluate_model(y_test, y_pred_xgb, y_pred_xgb, "XGBoost Classification")

In [ ]:
# --- Feature importance analysis ---
plt.figure(figsize=(10, 8))
feat_importances = pd.Series(xgb_clf.feature_importances_, index=X_train.columns)
feat_importances.nlargest(15).sort_values().plot(kind='barh', color='teal')
plt.title('Top 15 most important variables according to XGBoost model')
plt.xlabel('Importance level (Feature Importance)')
plt.show()

In [ ]:
# --- CONFUSION MATRIX ---
from sklearn.metrics import confusion_matrix

plt.figure(figsize=(10, 8))
# Creating the confusion matrix
cm = confusion_matrix(y_test, y_pred_xgb)
# Displaying as a heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)

plt.title('Confusion matrix for XGBoost model')
plt.xlabel('Predicted grade')
plt.ylabel('Actual grade')
plt.show()

## Student Grouping (K-Means Clustering)

In [ ]:
# Task: Splitting students into groups and analyzing profiles in the context of G3 results

In [ ]:
# Selection of key variables for segmentation based on prior feature importance analysis
cluster_features = ['failures', 'absences', 'Medu', 'studytime', 'higher']
X_cluster_train = X_train_scaled[cluster_features]
X_cluster_test = X_test_scaled[cluster_features]

In [ ]:
# --- Optimizing the number of clusters (Elbow Method) ---
inertia = []
K_range = range(1, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_cluster_train)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, marker='o', linestyle='--', color='navy')
plt.title('Elbow Method: Optimal number of clusters')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia (Within-cluster sum of squares)')
plt.show()

In [ ]:
# --- Implementation of target clustering model ---
# Chosen k=3 based on analysis of the inertia plot (inflection point)
kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=10)
train['cluster'] = kmeans.fit_predict(X_cluster_train)
test['cluster'] = kmeans.predict(X_cluster_test)

In [ ]:
# --- Profiling obtained groups (Clusters) ---
# Analysis of mean feature values in groups allows assigning descriptive labels to them
profiles = train.groupby('cluster')[cluster_features + ['G3']].mean().sort_values(by='G3', ascending=False)

In [ ]:
# NOTE: The order of groups may change depending on the seed.
# In the PDF report, describe the groups based on the obtained averages (e.g., high Medu = Leaders)
print("Group profiles in the training set (mean values):")
display(profiles)

In [ ]:
# --- Verification of division on the test set ---
plt.figure(figsize=(10, 6))
sns.boxplot(x='cluster', y='G3', data=test, palette='Set2', hue='cluster', legend=False)
plt.title('Relationship of group assignment with the final G3 grade (Test Set)')
plt.xlabel('Cluster number')
plt.ylabel('Grade G3')
plt.show()

In [ ]:
# Statistical evaluation of clustering effectiveness on test data
test_relation = test.groupby('cluster')['G3'].agg(['mean', 'std', 'count']).rename(columns={'mean': 'Mean G3', 'std': 'Std Dev', 'count': 'Count'})
print("\nStatistics of G3 grades for clusters on the TEST set:")
display(test_relation)

## Comparison and Conclusions

In [ ]:
# Consolidation of results from all modeling stages into a single summary table
summary_df = pd.DataFrame({
    'Model': ['Linear Regression (Estimation)', 'XGBoost (Classification)'],
    'Accuracy': [results_lin[0], results_xgb[0]],
    'Accuracy +/- 1 degree': [results_lin[1], results_xgb[1]],
    'MAE Error': [results_lin[2], results_xgb[2]]
})

print("MODEL QUALITY METRICS SUMMARY:")
display(summary_df)

In [ ]:
# Comparative visualization of MAE error
plt.figure(figsize=(10, 5))
sns.barplot(x='Model', y='MAE Error', data=summary_df, palette='viridis', hue='Model', legend=False)
plt.title('MAE error comparison (lower values mean better prediction)')
plt.ylim(0, max(summary_df['MAE Error']) * 1.2) # Axis scaling for better readability
plt.show()

print("\n--- FINAL ANALYSIS COMPLETED ---")

In [ ]:
# Saving datasets to CSV files for ZIP archive
train.to_csv('train_split.csv', index=False)
test.to_csv('test_split.csv', index=False)

In [ ]:
# Preparation of predictions file (according to ZIP requirement)
predictions = pd.DataFrame({
    'Actual_G3': y_test,
    'XGBoost_Class': y_pred_xgb,
    'LinearReg_Estimate': y_pred_reg_rounded,
    'Cluster_Assignment': test['cluster']
})
predictions.to_csv('predictions_and_groups.csv', index=False)